# Озеро ODS: март-дубли INT и май-объём trx

Две независимые проверки. Не смешивать в одном тикете.

**Блок 1** — тот же SQL, которым уже получили **666 739** дублей: секция **5f** в `excel_march_2026_chod_finrez_dip.ipynb` (`MEM_LIMIT=16g`, один месяц, периметр витрины).

**Блок 2** — объём `trx_cnt` / `trx_sum` в ODS за апр–май–июн (55% vs Excel этим SQL не считается).

**Блок 4** — точность схождения **строк** `inn+agr` (метрика 98% / 55%). Нужен готовый `final_df`, не ODS-тоталы.


In [ ]:
from calendar import monthrange

from IPython.display import display
import pandas as pd
from rail_connectors.connection import connect

MEM_LIMIT = '16g'

if 'imp' not in globals() or imp is None:
    imp = connect(
        to='IMPALA',
        extra_options={'db': 'sandbox_ai'},
        driver_args={'tez.queue.name': 'ai'},
        kerberos={
            'keytab_path': '/home/jovyan/test_requests/tech.keytab',
            'use_credentials': True,
            'update_keytab': True,
        },
        user_params={'user_name': 'Shestopalov-VYur'},
    )
    imp._init_connection()
    print('Impala connected')
else:
    print('Reuse existing imp')


def run_sql(sql, title=None):
    if title:
        print(title)
    with imp:
        imp.execute(f'set MEM_LIMIT={MEM_LIMIT}')
        df = imp.fetch(sql)
    display(df)
    return df


## 1) Дубли `n_amt_fee` — как в 5f (уже давало 666 739)

Не сырой `scd1_trx` × вся `trx_int`. Сначала узкие ключи витрины, потом fee только по ним:

1. `scd1_base24_fiids` — RSHB
2. `scd1_agreements` — SA, живые в месяце
3. `scd1_trx` — SA/S01, не reversed, `d_trx_orig` в месяце
4. `scd1_trx_acq` — есть `n_agr` из SA
5. `scd1_trx_int` — живые строки, `count(*)>1` и `count(distinct n_amt_fee)=1`

Один месяц за раз. Ожидание: март `multi_same_fee_dup` ≈ **666 739**, avg rows = 2; февраль ≈ 1; апрель ≈ 0.


In [ ]:
def month_bounds(ym):
    y, m = map(int, ym.split('-'))
    return f'{ym}-01', f'{ym}-{monthrange(y, m)[1]:02d}'


def cte_trx_keys(ms, me):
    # Как excel_march_2026_chod_finrez_dip.ipynb / секция 5f
    return f"""
    fiid_rshb as (
      select distinct cast(fa.c_fiid as string) as c_fiid
      from ods_alpha.scd1_base24_fiids fa
      where coalesce(cast(fa.c_fiid_grp as string), 'UNKNOWN') = 'RSHB'
    ),
    sa_agr as (
      select distinct cast(a.n_agr as string) as n_agr
      from ods_alpha.scd1_agreements a
      where upper(trim(cast(a.acq_class as string))) = 'SA'
        and cast(a.d_valid_from as date) <= cast('{me}' as date)
        and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{ms}' as date))
        and coalesce(a.ods_deleted_flg, '0') <> '1'
    ),
    trx_base as (
      select cast(t.n_trx as string) as n_trx
      from ods_alpha.scd1_trx t
      join fiid_rshb fr on fr.c_fiid = cast(t.c_fiid_acq as string)
      where cast(t.d_trx_orig as timestamp) >= cast('{ms}' as timestamp)
        and cast(t.d_trx_orig as timestamp) < cast(date_add(cast('{me}' as date), 1) as timestamp)
        and t.c_nter is not null
        and coalesce(t.ods_deleted_flg, '0') <> '1'
        and t.c_trx_class = 'SA'
        and t.c_trx_type = 'S01'
        and coalesce(t.cf_trx_stat, '') <> 'R'
      group by cast(t.n_trx as string)
    ),
    ta as (
      select cast(a.n_trx as string) as n_trx
      from ods_alpha.scd1_trx_acq a
      join trx_base tb on tb.n_trx = cast(a.n_trx as string)
      join sa_agr ss on ss.n_agr = cast(a.n_agr as string)
      where coalesce(a.ods_deleted_flg, '0') <> '1'
      group by cast(a.n_trx as string)
    )
    """


def sql_multi_class(ym):
    ms, me = month_bounds(ym)
    return f"""
    with {cte_trx_keys(ms, me)},
    int_alive as (
      select
        cast(i.n_trx as string) as n_trx,
        coalesce(cast(i.n_amt_fee as double), 0.0) as n_amt_fee
      from ods_alpha.scd1_trx_int i
      join ta k on k.n_trx = cast(i.n_trx as string)
      where coalesce(i.ods_deleted_flg, '0') <> '1'
    ),
    per_trx as (
      select
        n_trx,
        count(*) as int_rows,
        count(distinct cast(n_amt_fee as string)) as distinct_fee_values,
        sum(n_amt_fee) as fee_sum,
        max(n_amt_fee) as fee_max
      from int_alive
      group by n_trx
    ),
    multi as (
      select * from per_trx where int_rows > 1
    )
    select
      '{ym}' as report_month,
      count(*) as multi_trx_cnt,
      sum(case when distinct_fee_values = 1 then 1 else 0 end) as multi_same_fee_dup,
      sum(case when distinct_fee_values > 1 then 1 else 0 end) as multi_different_fees,
      avg(int_rows) as avg_rows_on_multi,
      sum(case when distinct_fee_values = 1 then fee_sum - fee_max else 0 end) as extra_fee_from_same_fee_dups
    from multi
    """


DUP_MONTHS = ['2026-02', '2026-03', '2026-04']
dup_parts = []
for ym in DUP_MONTHS:
    part = run_sql(sql_multi_class(ym), f'=== 1. Дубли trx_int (5f) {ym} ===')
    if part is not None and len(part):
        dup_parts.append(part)

march_by_month = pd.concat(dup_parts, ignore_index=True) if dup_parts else pd.DataFrame()
print('=== свод ===')
display(march_by_month)


Sample двух живых строк на один мартовский `n_trx` (тот же периметр `ta`).


In [ ]:
ms, me = month_bounds('2026-03')
sql_march_sample = f"""
with {cte_trx_keys(ms, me)},
multi as (
  select cast(i.n_trx as string) as n_trx
  from ods_alpha.scd1_trx_int i
  join ta k on k.n_trx = cast(i.n_trx as string)
  where coalesce(i.ods_deleted_flg, '0') <> '1'
  group by cast(i.n_trx as string)
  having count(*) > 1
     and count(distinct cast(i.n_amt_fee as string)) = 1
  limit 8
)
select
  cast(i.n_trx as string) as n_trx,
  cast(i.n_amt_fee as double) as n_amt_fee,
  cast(i.ods_deleted_flg as string) as ods_deleted_flg,
  cast(i.ods_op_type as string) as ods_op_type
from ods_alpha.scd1_trx_int i
join multi m on m.n_trx = cast(i.n_trx as string)
where coalesce(i.ods_deleted_flg, '0') <> '1'
order by 1, 2
"""

march_sample = run_sql(sql_march_sample, '=== 1b. Sample дублей март (периметр 5f) ===')


## 2) Май: количество и сумма транзакций в ODS

Если май здесь как апрель/июнь, а сверка с Excel 55% — смотреть Excel. Если май просел здесь — озеро.


In [ ]:
sql_may_trx_simple = r"""
select
  trunc(to_date(cast(t.d_trx_orig as timestamp)), 'MM') as trx_month,
  count(*) as rows_cnt,
  count(distinct t.n_trx) as trx_cnt,
  sum(cast(t.n_amt_src as double)) as trx_sum
from ods_alpha.scd1_trx t
where coalesce(t.ods_deleted_flg, '0') <> '1'
  and t.c_trx_class = 'SA'
  and t.c_trx_type = 'S01'
  and t.c_nter is not null
  and coalesce(t.cf_trx_stat, '') <> 'R'
  and cast(t.d_trx_orig as timestamp) >= cast('2026-04-01' as timestamp)
  and cast(t.d_trx_orig as timestamp) <  cast('2026-07-01' as timestamp)
group by 1
order by 1
"""

may_simple = run_sql(
    sql_may_trx_simple,
    '=== 2. ODS trx_cnt / trx_sum (только scd1_trx) ===',
)


Как витрина: RSHB + `trx_acq` на периметре месяца (как `ta` в 5f).


In [ ]:
def sql_may_volume_mart(ym):
    ms, me = month_bounds(ym)
    return f"""
    with {cte_trx_keys(ms, me)},
    trx_amt as (
      select
        cast(t.n_trx as string) as n_trx,
        max(cast(t.n_amt_src as double)) as n_amt_src
      from ods_alpha.scd1_trx t
      join ta k on k.n_trx = cast(t.n_trx as string)
      group by cast(t.n_trx as string)
    )
    select
      '{ym}' as report_month,
      count(*) as trx_cnt,
      sum(n_amt_src) as trx_sum
    from trx_amt
    """


vol_parts = []
for ym in ['2026-04', '2026-05', '2026-06']:
    part = run_sql(sql_may_volume_mart(ym), f'=== 2b. Объём как витрина {ym} ===')
    if part is not None and len(part):
        vol_parts.append(part)

may_mart = pd.concat(vol_parts, ignore_index=True) if vol_parts else pd.DataFrame()
print('=== свод апр / май / июн ===')
display(may_mart)


Май по дням: нет ли обрыва загрузки.


In [ ]:
sql_may_daily = r"""
select
  to_date(cast(t.d_trx_orig as timestamp)) as trx_dt,
  count(distinct t.n_trx) as trx_cnt,
  sum(cast(t.n_amt_src as double)) as trx_sum
from ods_alpha.scd1_trx t
where coalesce(t.ods_deleted_flg, '0') <> '1'
  and t.c_trx_class = 'SA'
  and t.c_trx_type = 'S01'
  and t.c_nter is not null
  and coalesce(t.cf_trx_stat, '') <> 'R'
  and cast(t.d_trx_orig as timestamp) >= cast('2026-05-01' as timestamp)
  and cast(t.d_trx_orig as timestamp) <  cast('2026-06-01' as timestamp)
group by 1
order by 1
"""

may_daily = run_sql(sql_may_daily, '=== 2c. Май по дням ===')
if may_daily is not None and len(may_daily):
    print(
        'дней с данными:', len(may_daily),
        '| min', may_daily['trx_dt'].min(),
        '| max', may_daily['trx_dt'].max(),
    )


## Что отправить озерщикам

**Тикет 1 — март.** SQL ячейки 1 (как 5f). Таблица `ods_alpha.scd1_trx_int`.

**Май:** сначала блок 3 (Excel vs ODS). Если тоталы сходятся во все месяцы, включая май — в тикет «пропали транзакции» не писать. 55% тогда только в сверке Excel ↔ `final_df`.


## 3) Есть ли такое же расхождение Excel vs озеро в других месяцах?

Озеро в мае по объёму не просело. Сверяем **тоталы** Excel (`Количество операций` / `Сумма операций`) с ODS:

- `lake_raw` — все SA/S01 (ячейка 2)
- `lake_mart` — как витрина, RSHB + `trx_acq` (ячейка 2b)

Считаем `ratio = lake_mart / excel` и `% расхождения`. Если май уникально ~0.55, а соседние месяцы ~1.0 — ломается Excel/сверка мая, не загрузка ODS. Если все месяцы одинаково кривые — смотреть методику колонок.


In [ ]:
import re
from pathlib import Path

import numpy as np

DATA_DIR = Path('/home/jovyan/documents/Equaring/Data')
EXCEL_BY_MONTH = {
    '2026-01': (DATA_DIR / '01_Январь_2026.xlsx', 1),
    '2026-02': (DATA_DIR / '02_Февраль_2026.xlsx', 1),
    '2026-03': (DATA_DIR / '03_Март_2026.xlsx', 0),
    '2026-04': (DATA_DIR / '04_Апрель_2026.xlsx', 0),
    '2026-05': (DATA_DIR / '05_Май_2026.xlsx', 0),
    '2026-06': (DATA_DIR / '06_Июнь_2026.xlsx', 0),
}
TRX_CNT_CAND = ['Количество операций', 'Количеств операций', 'trx_cnt']
TRX_SUM_CAND = ['Сумма операций', 'Сумма опреаций', 'trx_sum']
INN_CAND = ['ИНН', 'inn', 'c_inn']
AGR_CAND = ['ID договора', 'Номер договора', 'agr_id', 'abs_agr_id']


def _pick_col(columns, candidates):
    cols = list(columns)
    norm = lambda s: re.sub(r'\s+', ' ', str(s).replace('\xa0', ' ').strip().lower())
    norm_map = {norm(c): c for c in cols}
    for c in candidates:
        if c in cols:
            return c
        if norm(c) in norm_map:
            return norm_map[norm(c)]
    return None


def _to_num(s):
    return pd.to_numeric(
        s.astype(str).str.replace('\xa0', '', regex=False).str.replace(' ', '', regex=False).str.replace(',', '.', regex=False),
        errors='coerce',
    )


def _norm_inn(v):
    if pd.isna(v):
        return None
    s = re.sub(r'\D+', '', re.sub(r'\.0$', '', str(v).strip()))
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)
    return s if len(s) in (10, 12) else None


def _norm_agr(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', '').replace(' ', '')
    return s or None


excel_rows = []
for ym, (path, header) in EXCEL_BY_MONTH.items():
    if not path.exists():
        print(f'Excel {ym}: нет файла {path}')
        continue
    ex = pd.read_excel(path, header=header)
    c_cnt = _pick_col(ex.columns, TRX_CNT_CAND)
    c_sum = _pick_col(ex.columns, TRX_SUM_CAND)
    c_inn = _pick_col(ex.columns, INN_CAND)
    c_agr = _pick_col(ex.columns, AGR_CAND)
    print(f'Excel {ym}: cnt={c_cnt!r} sum={c_sum!r} rows={len(ex):,}')
    if c_cnt is None or c_sum is None:
        continue
    cnt = _to_num(ex[c_cnt])
    sm = _to_num(ex[c_sum])
    keys = None
    dup_share = np.nan
    if c_inn and c_agr:
        inn = ex[c_inn].map(_norm_inn)
        agr = ex[c_agr].map(_norm_agr)
        keys = inn.astype(str) + '|' + agr.astype(str)
        valid = inn.notna() & agr.notna()
        vc = keys[valid].value_counts()
        dup_share = float((vc > 1).mean()) if len(vc) else 0.0
        # как в боевом QC: count = max, sum = sum по inn+agr
        tmp = pd.DataFrame({'k': keys, 'cnt': cnt, 'sm': sm})
        tmp = tmp.loc[valid]
        g = tmp.groupby('k', as_index=False).agg(cnt=('cnt', 'max'), sm=('sm', 'sum'))
        excel_cnt = float(g['cnt'].fillna(0).sum())
        excel_sum = float(g['sm'].fillna(0).sum())
    else:
        excel_cnt = float(cnt.fillna(0).sum())
        excel_sum = float(sm.fillna(0).sum())
    excel_rows.append({
        'report_month': ym,
        'excel_trx_cnt': excel_cnt,
        'excel_trx_sum': excel_sum,
        'excel_dup_key_share': dup_share,
        'excel_rows': len(ex),
    })

excel_tot = pd.DataFrame(excel_rows)
print('=== Excel тоталы (inn+agr: max cnt / sum sum) ===')
display(excel_tot)

# ODS raw Jan–Jun, если в памяти только апр–июн
need_raw_months = set(excel_tot['report_month'])
have_raw = set()
if 'may_simple' in globals() and may_simple is not None and len(may_simple):
    raw = may_simple.copy()
    raw['report_month'] = pd.to_datetime(raw['trx_month']).dt.strftime('%Y-%m')
    have_raw = set(raw['report_month'])
else:
    raw = pd.DataFrame(columns=['report_month', 'trx_cnt', 'trx_sum'])

missing_raw = sorted(need_raw_months - have_raw)
if missing_raw:
    start = f'{missing_raw[0]}-01'
    end_y, end_m = map(int, missing_raw[-1].split('-'))
    end_m += 1
    end_y += (end_m - 1) // 12
    end_m = (end_m - 1) % 12 + 1
    end = f'{end_y:04d}-{end_m:02d}-01'
    sql_raw_extra = f"""
    select
      trunc(to_date(cast(t.d_trx_orig as timestamp)), 'MM') as trx_month,
      count(distinct t.n_trx) as trx_cnt,
      sum(cast(t.n_amt_src as double)) as trx_sum
    from ods_alpha.scd1_trx t
    where coalesce(t.ods_deleted_flg, '0') <> '1'
      and t.c_trx_class = 'SA'
      and t.c_trx_type = 'S01'
      and t.c_nter is not null
      and coalesce(t.cf_trx_stat, '') <> 'R'
      and cast(t.d_trx_orig as timestamp) >= cast('{start}' as timestamp)
      and cast(t.d_trx_orig as timestamp) <  cast('{end}' as timestamp)
    group by 1
    """
    extra_raw = run_sql(sql_raw_extra, f'=== ODS raw догрузка {missing_raw} ===')
    extra_raw = extra_raw.copy()
    extra_raw['report_month'] = pd.to_datetime(extra_raw['trx_month']).dt.strftime('%Y-%m')
    raw = pd.concat([raw, extra_raw], ignore_index=True)
    raw = raw.drop_duplicates('report_month', keep='last')

# mart: reuse 2b, добрать недостающие месяцы тем же cte_trx_keys
have_mart = set()
if 'may_mart' in globals() and may_mart is not None and len(may_mart):
    mart = may_mart.copy()
    mart['report_month'] = mart['report_month'].astype(str).str[:7]
    have_mart = set(mart['report_month'])
else:
    mart = pd.DataFrame(columns=['report_month', 'trx_cnt', 'trx_sum'])

for ym in excel_tot['report_month']:
    if ym in have_mart:
        continue
    if 'sql_may_volume_mart' not in globals():
        print(f'Пропуск mart {ym}: сначала выполни ячейку 2b (sql_may_volume_mart)')
        continue
    part = run_sql(sql_may_volume_mart(ym), f'=== mart догрузка {ym} ===')
    if part is not None and len(part):
        mart = pd.concat([mart, part], ignore_index=True)
        have_mart.add(ym)

cmp = excel_tot.merge(
    raw[['report_month', 'trx_cnt', 'trx_sum']].rename(columns={'trx_cnt': 'lake_raw_cnt', 'trx_sum': 'lake_raw_sum'}),
    on='report_month',
    how='left',
).merge(
    mart[['report_month', 'trx_cnt', 'trx_sum']].rename(columns={'trx_cnt': 'lake_mart_cnt', 'trx_sum': 'lake_mart_sum'}),
    on='report_month',
    how='left',
)

for lake_c, excel_c, out in [
    ('lake_mart_cnt', 'excel_trx_cnt', 'ratio_mart_excel_cnt'),
    ('lake_mart_sum', 'excel_trx_sum', 'ratio_mart_excel_sum'),
    ('lake_raw_cnt', 'excel_trx_cnt', 'ratio_raw_excel_cnt'),
    ('lake_raw_sum', 'excel_trx_sum', 'ratio_raw_excel_sum'),
]:
    cmp[out] = np.where(cmp[excel_c].fillna(0) == 0, np.nan, cmp[lake_c] / cmp[excel_c])

cmp['div_pct_mart_cnt'] = (1 - cmp['ratio_mart_excel_cnt']).abs() * 100
cmp = cmp.sort_values('report_month').reset_index(drop=True)

print('=== Excel vs ODS по месяцам ===')
print('ratio ≈ 1 — тоталы сходятся; май ~0.55 при остальных ~1 — ломается Excel/сверка мая')
display(cmp)

print('=== ratio_mart_excel_cnt по месяцам ===')
display(cmp[['report_month', 'excel_trx_cnt', 'lake_mart_cnt', 'ratio_mart_excel_cnt', 'div_pct_mart_cnt', 'excel_dup_key_share']])


## 4) Точность схождения строк (та же метрика, что 98% / 55%)

Зерно: `inn + agr_id` (как в `01_07_acq_dash_jan_jun_mpos`, QC row exact match).

Считается только на ключах, которые есть **и в Excel, и в `final_df`**.  
`trx_cnt` — точное равенство; `trx_sum` — допуск 0.01.

Озеро берём из уже посчитанного `final_df` (CSV или checkpoint), **без Impala**. Если файлов нет — ячейка скажет, что загрузить.


In [ ]:
import re
from decimal import Decimal, InvalidOperation
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

DATA_DIR = Path('/home/jovyan/documents/Equaring/Data')
MONTHS = ['2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06']
EXCEL_BY_MONTH = {
    '2026-01': (DATA_DIR / '01_Январь_2026.xlsx', 1),
    '2026-02': (DATA_DIR / '02_Февраль_2026.xlsx', 1),
    '2026-03': (DATA_DIR / '03_Март_2026.xlsx', 0),
    '2026-04': (DATA_DIR / '04_Апрель_2026.xlsx', 0),
    '2026-05': (DATA_DIR / '05_Май_2026.xlsx', 0),
    '2026-06': (DATA_DIR / '06_Июнь_2026.xlsx', 0),
}
PERIOD_CSVS = [
    DATA_DIR / 'final_df_period_2026_01_2026_07_mpos.csv',
    DATA_DIR / 'final_df_period_2026_01_2026_06_mpos.csv',
]
CKPT_DIRS = [
    DATA_DIR / 'checkpoints_final_df_2026_01_2026_06_mpos',
    DATA_DIR / 'checkpoints_final_df_2026_01_2026_06',
]
READY_QC = [
    DATA_DIR / 'row_exact_match_2026_01_2026_07_mpos.csv',
    DATA_DIR / 'row_exact_match_2026_01_2026_06_mpos.csv',
]
MONEY_TOL = 0.01
INN_CAND = ['ИНН', 'inn', 'c_inn']
AGR_CAND = ['ID договора', 'Номер договора', 'agr_id', 'abs_agr_id']
CNT_CAND = ['Количество операций', 'Количеств операций', 'trx_cnt']
SUM_CAND = ['Сумма операций', 'Сумма опреаций', 'trx_sum']


def normalize_inn_q1(v):
    if pd.isna(v):
        return None
    s = re.sub(r'\D+', '', re.sub(r'\.0$', '', str(v).strip()))
    if not s:
        return None
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)
    return s if len(s) in (10, 12) else None


def normalize_agr_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', '').replace(' ', '').replace(',', '.')
    if s in {'', 'nan', 'None'}:
        return None
    try:
        d = Decimal(s)
        if d == d.to_integral_value():
            return str(int(d))
    except (InvalidOperation, ValueError):
        pass
    s = re.sub(r'\.0$', '', s)
    return s if s not in {'', 'nan', 'None'} else None


def _pick_col(columns, candidates):
    cols = list(columns)
    norm = lambda x: re.sub(r'\s+', ' ', str(x).replace('\xa0', ' ').strip().lower())
    norm_map = {norm(c): c for c in cols}
    for c in candidates:
        if c in cols:
            return c
        if norm(c) in norm_map:
            return norm_map[norm(c)]
    return None


def _to_num(s):
    return pd.to_numeric(
        s.astype(str).str.replace('\xa0', '', regex=False).str.replace(' ', '', regex=False).str.replace(',', '.', regex=False),
        errors='coerce',
    )


def _month_key(s):
    t = pd.to_datetime(s, errors='coerce')
    if pd.isna(t):
        s = str(s)
        return s[:7] if len(s) >= 7 else s
    return t.strftime('%Y-%m')


# 0) уже готовый QC — только показать
for p in READY_QC:
    if p.exists():
        ready = pd.read_csv(p)
        print(f'Уже есть готовый QC: {p}')
        piv = ready.pivot_table(index='report_month', columns='metric', values='exact_match_pct', aggfunc='first')
        display(piv)
        print('Это прошлый прогон. Ниже пересчёт из final_df + Excel (если файлы есть).')
        break

# 1) найти final_df
period_df = None
period_src = None
if 'final_df_period_df' in globals() and final_df_period_df is not None and len(final_df_period_df):
    period_df = final_df_period_df
    period_src = 'memory:final_df_period_df'
else:
    for p in PERIOD_CSVS:
        if p.exists():
            period_df = pd.read_csv(p, dtype={'inn': 'string', 'agr_id': 'string', 'n_agr': 'string'}, low_memory=False)
            period_src = str(p)
            break

print('period_df:', period_src, 'rows=' + (f'{len(period_df):,}' if period_df is not None else 'none'))
for d in CKPT_DIRS:
    print(f'ckpt {d.name}: exists={d.exists()} files={len(list(d.glob("final_df_*"))) if d.exists() else 0}')


def load_lake_month(month_label):
    if 'final_df_by_month' in globals() and final_df_by_month:
        df = final_df_by_month.get(month_label)
        if df is not None and len(df):
            print(f'  lake {month_label}: memory final_df_by_month rows={len(df):,}')
            return df
    if period_df is not None:
        col = 'report_month' if 'report_month' in period_df.columns else (
            'snapshot_month_start' if 'snapshot_month_start' in period_df.columns else None
        )
        if col:
            out = period_df.loc[period_df[col].map(_month_key) == month_label].copy()
            if len(out):
                print(f'  lake {month_label}: period CSV rows={len(out):,}')
                return out
    month_us = month_label.replace('-', '_')
    for d in CKPT_DIRS:
        if not d.exists():
            continue
        for name in [f'final_df_{month_us}.parquet', f'final_df_{month_label}.parquet', f'final_df_{month_us}.csv']:
            p = d / name
            if not p.exists():
                continue
            out = pd.read_parquet(p) if p.suffix == '.parquet' else pd.read_csv(p, dtype={'inn': 'string', 'agr_id': 'string'})
            print(f'  lake {month_label}: {d.name}/{p.name} rows={len(out):,}')
            return out
    return None


def lake_agg(df):
    lk = df.copy()
    if 'agr_id' not in lk.columns and 'n_agr' in lk.columns:
        lk['agr_id'] = lk['n_agr']
    lk['inn_key'] = lk['inn'].map(normalize_inn_q1)
    lk['agr_id_key'] = lk['agr_id'].map(normalize_agr_q1)
    lk['trx_cnt_lake'] = pd.to_numeric(lk['trx_cnt'], errors='coerce')
    lk['trx_sum_lake'] = pd.to_numeric(lk['trx_sum'], errors='coerce')
    return (
        lk.dropna(subset=['inn_key', 'agr_id_key'])
          .groupby(['inn_key', 'agr_id_key'], as_index=False)
          .agg(trx_cnt_lake=('trx_cnt_lake', 'max'), trx_sum_lake=('trx_sum_lake', 'max'))
    )


def excel_agg(path, header):
    ex = pd.read_excel(path, header=header)
    c_inn = _pick_col(ex.columns, INN_CAND)
    c_agr = _pick_col(ex.columns, AGR_CAND)
    c_cnt = _pick_col(ex.columns, CNT_CAND)
    c_sum = _pick_col(ex.columns, SUM_CAND)
    if None in (c_inn, c_agr, c_cnt, c_sum):
        raise ValueError(f'колонки: inn={c_inn} agr={c_agr} cnt={c_cnt} sum={c_sum} available={list(ex.columns)}')
    ex['inn_key'] = ex[c_inn].map(normalize_inn_q1)
    ex['agr_id_key'] = ex[c_agr].map(normalize_agr_q1)
    ex['trx_cnt_excel'] = _to_num(ex[c_cnt])
    ex['trx_sum_excel'] = _to_num(ex[c_sum])
    return (
        ex.dropna(subset=['inn_key', 'agr_id_key'])
          .groupby(['inn_key', 'agr_id_key'], as_index=False)
          .agg(trx_cnt_excel=('trx_cnt_excel', 'max'), trx_sum_excel=('trx_sum_excel', 'sum'))
    )


parts = []
missing_lake = []
for month in MONTHS:
    print(f'\n=== row match {month} ===')
    xl = EXCEL_BY_MONTH[month]
    if not xl[0].exists():
        print(f'  Excel нет: {xl[0]}')
        continue
    lake = load_lake_month(month)
    if lake is None or lake.empty:
        print(f'  final_df нет — skip. Нужен CSV/checkpoint или ячейка QC в 01_07_acq_dash_jan_jun_mpos')
        missing_lake.append(month)
        continue
    la = lake_agg(lake)
    ea = excel_agg(xl[0], xl[1])
    cmp = la.merge(ea, on=['inn_key', 'agr_id_key'], how='outer', indicator=True)
    n_both = int((cmp['_merge'] == 'both').sum())
    n_xl = int((cmp['_merge'] == 'right_only').sum())
    n_lk = int((cmp['_merge'] == 'left_only').sum())
    print(f'  keys both={n_both:,} only_excel={n_xl:,} only_lake={n_lk:,}')
    both = cmp.loc[cmp['_merge'] == 'both']
    for metric, lcol, ecol, tol in [
        ('trx_cnt', 'trx_cnt_lake', 'trx_cnt_excel', 0.0),
        ('trx_sum', 'trx_sum_lake', 'trx_sum_excel', MONEY_TOL),
    ]:
        lv = pd.to_numeric(both[lcol], errors='coerce').fillna(0.0)
        ev = pd.to_numeric(both[ecol], errors='coerce').fillna(0.0)
        exact = np.isclose(lv, ev, atol=tol, rtol=0) if tol else lv.eq(ev)
        exact_n = int(exact.sum())
        pct = round(100.0 * exact_n / n_both, 2) if n_both else None
        parts.append({
            'report_month': month,
            'metric': metric,
            'keys_both': n_both,
            'keys_only_excel': n_xl,
            'keys_only_lake': n_lk,
            'exact_match_n': exact_n,
            'exact_match_pct': pct,
            'mismatch_n': n_both - exact_n,
        })
        print(f'  {metric}: exact={pct}% ({exact_n:,}/{n_both:,}) mismatch={n_both - exact_n:,}')

if missing_lake and not parts:
    print('\nНет final_df ни за один месяц.')
    print('Вариант А: в 01_07_acq_dash_jan_jun_mpos.ipynb поставь')
    print('  force_recompute_final_df=False, wipe_checkpoints_on_force=False,')
    print('  run_excel_qc=True, run_drp_upload=False')
    print('и запусти QC «точное совпадение строк» (без месячного цикла Impala).')
    print('Вариант Б: положи parquet в checkpoints_final_df_2026_01_2026_06:')
    print('  final_df_2026_01.parquet … final_df_2026_06.parquet')
elif parts:
    row_exact_match_df = pd.DataFrame(parts)
    pivot = row_exact_match_df.pivot_table(
        index='report_month', columns='metric', values='exact_match_pct', aggfunc='first'
    )
    cov = (
        row_exact_match_df.groupby('report_month', as_index=False)
        .agg(keys_both=('keys_both', 'first'), keys_only_excel=('keys_only_excel', 'first'), keys_only_lake=('keys_only_lake', 'first'))
    )
    print('\n=== покрытие ключей inn+agr ===')
    display(cov)
    print('=== exact_match_pct (это 98% / 55%) ===')
    display(pivot)
    out = DATA_DIR / 'row_exact_match_trx_2026_01_2026_06.csv'
    row_exact_match_df.to_csv(out, index=False, encoding='utf-8-sig')
    print('saved', out)
    if missing_lake:
        print('нет final_df за:', missing_lake)


## 5) Причина мая: разложить 2 863 расхождения

Уже ясно из блока 4 (не путать с другими колонками):

| Что | Вывод |
|---|---|
| `keys_both` май = 6 451, only_excel=17, only_lake=13 | Строки не пропали. Это не дыра в озере. |
| `retl_cnt` 100%, `term_cnt` 99.98%, `commission_monthly` 99.52% | Договор, точки, MPOS-аренда — те же, что в Excel. |
| `trx_cnt` 55.62% и `trx_sum` 55.60% (2 863 / 2 864) | Ломается **одна и та же** пачка ключей, оба поля операций. |
| Тоталы мая ближе к Excel (≈1.2%), чем янв–апр (≈4%) | Объём месяца не «вдвое меньше». Значения по строкам разъехались и **компенсируются** в сумме. |

`amortization` 33–40% **во все месяцы** — другая история: озеро = модель `shestopalov_terminal_amortization_model`, Excel = своя «Амортизация». `fin_result` тянется за ней (`chod − aur − amort`). Февральский провал `aur` 70% — третья история (`aur = retl_cnt × 1926`).

Ячейка ниже **не ходит в Impala**. Классифицирует майские mismatch: знак, ratio, доля объёма, сверка с апрелем/июнем, `commission_from_ops` (тот же section 05).


In [ ]:
TARGET = '2026-05'
CTRL = ['2026-04', '2026-06']
OUT = DATA_DIR / 'debug_trx_may_2026'
OUT.mkdir(parents=True, exist_ok=True)

COMM_OPS_CAND = [
    'Комиссия эквайринга', 'Комиссия (% с операций)', 'Комиссия \n(% с операций)',
    'Комиссия % с операций', 'commission_from_ops',
]


def _excel_full(path, header):
    ex = pd.read_excel(path, header=header)
    c_inn = _pick_col(ex.columns, INN_CAND)
    c_agr = _pick_col(ex.columns, AGR_CAND)
    c_cnt = _pick_col(ex.columns, CNT_CAND)
    c_sum = _pick_col(ex.columns, SUM_CAND)
    c_ops = _pick_col(ex.columns, COMM_OPS_CAND)
    print(f'Excel {path.name}: header={header} rows={len(ex):,} cols={len(ex.columns)}')
    print('  mapped:', {'inn': c_inn, 'agr': c_agr, 'trx_cnt': c_cnt, 'trx_sum': c_sum, 'comm_ops': c_ops})
    print('  all columns:', list(ex.columns))
    out = pd.DataFrame({
        'inn_key': ex[c_inn].map(normalize_inn_q1),
        'agr_id_key': ex[c_agr].map(normalize_agr_q1),
        'trx_cnt_excel': _to_num(ex[c_cnt]),
        'trx_sum_excel': _to_num(ex[c_sum]),
    })
    if c_ops:
        out['commission_from_ops_excel'] = _to_num(ex[c_ops])
    else:
        out['commission_from_ops_excel'] = np.nan
    return (
        out.dropna(subset=['inn_key', 'agr_id_key'])
           .groupby(['inn_key', 'agr_id_key'], as_index=False)
           .agg(
               trx_cnt_excel=('trx_cnt_excel', 'max'),
               trx_sum_excel=('trx_sum_excel', 'sum'),
               commission_from_ops_excel=('commission_from_ops_excel', 'sum'),
               excel_rows=('trx_cnt_excel', 'size'),
           )
    )


def _lake_full(month):
    df = load_lake_month(month)
    if df is None or df.empty:
        raise RuntimeError(f'no final_df for {month}')
    lk = df.copy()
    if 'agr_id' not in lk.columns and 'n_agr' in lk.columns:
        lk['agr_id'] = lk['n_agr']
    lk['inn_key'] = lk['inn'].map(normalize_inn_q1)
    lk['agr_id_key'] = lk['agr_id'].map(normalize_agr_q1)
    for c in ['trx_cnt', 'trx_sum', 'commission_from_ops']:
        if c not in lk.columns:
            lk[c] = np.nan
        lk[c] = pd.to_numeric(lk[c], errors='coerce')
    return (
        lk.dropna(subset=['inn_key', 'agr_id_key'])
          .groupby(['inn_key', 'agr_id_key'], as_index=False)
          .agg(
              trx_cnt_lake=('trx_cnt', 'max'),
              trx_sum_lake=('trx_sum', 'max'),
              commission_from_ops_lake=('commission_from_ops', 'max'),
              lake_rows=('trx_cnt', 'size'),
          )
    )


def _both(month):
    path, header = EXCEL_BY_MONTH[month]
    ea = _excel_full(path, header)
    la = _lake_full(month)
    cmp = la.merge(ea, on=['inn_key', 'agr_id_key'], how='outer', indicator=True)
    both = cmp.loc[cmp['_merge'] == 'both'].copy()
    both['trx_cnt_delta'] = both['trx_cnt_lake'] - both['trx_cnt_excel']
    both['trx_sum_delta'] = both['trx_sum_lake'] - both['trx_sum_excel']
    both['trx_cnt_ratio'] = np.where(both['trx_cnt_excel'].fillna(0) == 0, np.nan, both['trx_cnt_lake'] / both['trx_cnt_excel'])
    both['trx_sum_ratio'] = np.where(both['trx_sum_excel'].fillna(0) == 0, np.nan, both['trx_sum_lake'] / both['trx_sum_excel'])
    both['cnt_exact'] = both['trx_cnt_lake'].fillna(0).eq(both['trx_cnt_excel'].fillna(0))
    both['sum_exact'] = np.isclose(both['trx_sum_lake'].fillna(0), both['trx_sum_excel'].fillna(0), atol=0.01, rtol=0)
    ops_ok = both['commission_from_ops_excel'].notna().any()
    if ops_ok:
        both['ops_exact'] = np.isclose(
            both['commission_from_ops_lake'].fillna(0),
            both['commission_from_ops_excel'].fillna(0),
            atol=0.01, rtol=0,
        )
    else:
        both['ops_exact'] = False
    print(
        f'{month}: both={len(both):,} only_excel={(cmp["_merge"]=="right_only").sum():,} '
        f'only_lake={(cmp["_merge"]=="left_only").sum():,} excel_dup_keys={(ea["excel_rows"]>1).sum():,}'
    )
    return both, cmp, ops_ok


may_both, may_cmp, may_ops = _both(TARGET)
ctrl = {}
for m in CTRL:
    if EXCEL_BY_MONTH[m][0].exists():
        ctrl[m] = _both(m)[0]

mis = may_both.loc[~may_both['cnt_exact']].copy()
ok = may_both.loc[may_both['cnt_exact']].copy()
print(f'\nMay mismatches trx_cnt={len(mis):,}  exact={len(ok):,}')


def _share(df, col):
    return float(df[col].fillna(0).sum())


print('\n=== A. Объём на exact vs mismatch ===')
vol = pd.DataFrame([
    {
        'slice': 'exact keys',
        'keys': len(ok),
        'lake_trx_cnt': _share(ok, 'trx_cnt_lake'),
        'excel_trx_cnt': _share(ok, 'trx_cnt_excel'),
        'lake_trx_sum': _share(ok, 'trx_sum_lake'),
        'excel_trx_sum': _share(ok, 'trx_sum_excel'),
    },
    {
        'slice': 'mismatch keys',
        'keys': len(mis),
        'lake_trx_cnt': _share(mis, 'trx_cnt_lake'),
        'excel_trx_cnt': _share(mis, 'trx_cnt_excel'),
        'lake_trx_sum': _share(mis, 'trx_sum_lake'),
        'excel_trx_sum': _share(mis, 'trx_sum_excel'),
    },
    {
        'slice': 'all both',
        'keys': len(may_both),
        'lake_trx_cnt': _share(may_both, 'trx_cnt_lake'),
        'excel_trx_cnt': _share(may_both, 'trx_cnt_excel'),
        'lake_trx_sum': _share(may_both, 'trx_sum_lake'),
        'excel_trx_sum': _share(may_both, 'trx_sum_excel'),
    },
])
for c in ['lake_trx_cnt', 'excel_trx_cnt', 'lake_trx_sum', 'excel_trx_sum']:
    vol[c + '_share_pct'] = 100.0 * vol[c] / vol.loc[vol['slice']=='all both', c].iloc[0]
vol['cnt_ratio_l_e'] = vol['lake_trx_cnt'] / vol['excel_trx_cnt'].replace(0, np.nan)
display(vol)


print('\n=== B. Знак и бакеты ratio (trx_cnt) ===')
lake_gt = int((mis['trx_cnt_delta'] > 0).sum())
excel_gt = int((mis['trx_cnt_delta'] < 0).sum())
print(f'lake > excel: {lake_gt:,} | excel > lake: {excel_gt:,}')


def _bucket(r):
    if pd.isna(r):
        return 'excel=0 (ratio na)'
    if abs(r - 0.5) < 0.03:
        return '~0.5 (озеро ≈ половина Excel)'
    if abs(r - 2.0) < 0.06:
        return '~2 (озеро ≈ 2× Excel)'
    if 0.95 <= r <= 1.05:
        return '~1 ±5% (почти, но не exact)'
    if r < 0.5:
        return '<0.5'
    if r < 1:
        return '0.5–0.95'
    if r < 2:
        return '1.05–2'
    return '>2'


mis['ratio_bucket'] = mis['trx_cnt_ratio'].map(_bucket)
mis.loc[mis['trx_cnt_lake'].fillna(0).eq(0) & mis['trx_cnt_excel'].fillna(0).gt(0), 'ratio_bucket'] = 'lake=0 excel>0'
bkt = (
    mis.groupby('ratio_bucket', as_index=False)
       .agg(keys=('inn_key', 'size'), lake_cnt=('trx_cnt_lake', 'sum'), excel_cnt=('trx_cnt_excel', 'sum'))
       .sort_values('keys', ascending=False)
)
display(bkt)

print('\n=== C. commission_from_ops на тех же ключах (section 05) ===')
if may_ops:
    ops_pct = 100.0 * may_both['ops_exact'].mean()
    ops_on_mis = 100.0 * mis['ops_exact'].mean()
    print(f'ops exact all both = {ops_pct:.2f}%')
    print(f'ops exact на trx_cnt-mismatch = {ops_on_mis:.2f}%')
    if ops_pct >= 95:
        print('→ Комиссия с операций сходится, счётчики операций — нет. Excel «Количество/Сумма операций» за май расходится с тем же источником, что n_amt_tax.')
    elif ops_pct < 80:
        print('→ Ломается весь контур trx Excel vs section 05 (не только подпись колонки cnt/sum).')
else:
    print('В Excel мая нет колонки комиссии с операций — этот тест пропущен.')


print('\n=== D. Не тот месяц? Excel мая == апрель/июнь ===')

def _eq_rate(a, b):
    m = a.notna() & b.notna()
    if m.sum() == 0:
        return None, 0
    return round(100.0 * (a[m] == b[m]).mean(), 2), int(m.sum())

for other_label, other_df, other_col, may_col in [
    ('excel May == excel Apr', ctrl.get('2026-04'), 'trx_cnt_excel', 'trx_cnt_excel'),
    ('excel May == lake Apr', ctrl.get('2026-04'), 'trx_cnt_lake', 'trx_cnt_excel'),
    ('lake May == excel Apr', ctrl.get('2026-04'), 'trx_cnt_excel', 'trx_cnt_lake'),
    ('excel May == excel Jun', ctrl.get('2026-06'), 'trx_cnt_excel', 'trx_cnt_excel'),
    ('excel May == lake Jun', ctrl.get('2026-06'), 'trx_cnt_lake', 'trx_cnt_excel'),
]:
    if other_df is None:
        print(f'  {other_label}: нет {other_label[-3:]}')
        continue
    j = mis.merge(
        other_df[['inn_key', 'agr_id_key', other_col]].rename(columns={other_col: '_ref'}),
        on=['inn_key', 'agr_id_key'], how='inner',
    )
    pct, n = _eq_rate(j[may_col], j['_ref'])
    print(f'  {other_label}: {pct}% совпадений на {n:,} mismatch-ключах')


print('\n=== E. TOP-20 mismatch по |delta trx_cnt| ===')
top = mis.assign(abs_d=mis['trx_cnt_delta'].abs()).sort_values('abs_d', ascending=False)
cols = [
    'inn_key', 'agr_id_key',
    'trx_cnt_lake', 'trx_cnt_excel', 'trx_cnt_delta', 'trx_cnt_ratio',
    'trx_sum_lake', 'trx_sum_excel', 'trx_sum_ratio',
    'commission_from_ops_lake', 'commission_from_ops_excel',
]
display(top[cols].head(20))
top[cols].head(300).to_csv(OUT / 'may_trx_mismatch_top300.csv', index=False, encoding='utf-8-sig')
bkt.to_csv(OUT / 'may_trx_mismatch_ratio_buckets.csv', index=False, encoding='utf-8-sig')
vol.to_csv(OUT / 'may_trx_mismatch_volume_split.csv', index=False, encoding='utf-8-sig')
print('saved', OUT)


print('\n=== AUTO VERDICT ===')
dom = bkt.iloc[0]['ratio_bucket'] if len(bkt) else None
print('Доминирующий бакет:', dom)
if lake_gt > 0 and excel_gt > 0 and min(lake_gt, excel_gt) / max(lake_gt, excel_gt) > 0.4:
    print('Знак смешанный: часть договоров озеро больше, часть Excel больше — типично для разной методики/перекладки объёма, не для «пропал месяц».')
if vol.loc[vol['slice']=='mismatch keys', 'cnt_ratio_l_e'].iloc[0] and abs(vol.loc[vol['slice']=='all both', 'cnt_ratio_l_e'].iloc[0] - 1) < 0.05:
    print('Тотал both ≈ 1, а строки нет: расхождения компенсируются. Искать не дыру ODS, а правило, как Excel размазал операции по договорам в мае.')
if dom and '2' in str(dom):
    print('Похоже на двойной учёт с одной стороны (или Excel без одного из контуров).')
if dom and '0.5' in str(dom):
    print('Похоже, Excel мая считает шире (два контура / YTD / обе стороны), озеро — уже.')


## 6) Май: `max` по ключу или реальный недобор section 05?

Блок 5 уже закрыл: не чужой файл, не ×2, не дыра ключей. Excel больше на **2 854** из 2 863 mismatch, почти все в бакете **~1±5%**. Объём mismatch ≈ 89%, недобор озера на них ≈ **2.6%** (172 тыс. операций).

Две оставшиеся причины:

1. В `final_df` на один `inn+agr` несколько строк (два `n_agr` / дубль джойна) — QC берёт **`max`**, Excel — одну полную цифру. Тогда `sum` вместо `max` поднимет exact с 55% к ~99%.
2. На ключе одна строка, и в ней озеро правда меньше — Excel мая шире, или section 05 уже отрезал кусок (до `final_df`).

Ячейка без Impala. Если (1) не подтвердится — дальше SQL с ослаблением фильтров 05.


In [ ]:
def _lake_raw_grain(month):
    df = load_lake_month(month)
    if df is None or df.empty:
        raise RuntimeError(f'no final_df {month}')
    lk = df.copy()
    if 'agr_id' not in lk.columns and 'n_agr' in lk.columns:
        lk['agr_id'] = lk['n_agr']
    lk['inn_key'] = lk['inn'].map(normalize_inn_q1)
    lk['agr_id_key'] = lk['agr_id'].map(normalize_agr_q1)
    lk['trx_cnt'] = pd.to_numeric(lk['trx_cnt'], errors='coerce')
    lk['trx_sum'] = pd.to_numeric(lk['trx_sum'], errors='coerce')
    if 'commission_from_ops' in lk.columns:
        lk['commission_from_ops'] = pd.to_numeric(lk['commission_from_ops'], errors='coerce')
    else:
        lk['commission_from_ops'] = np.nan
    lk = lk.dropna(subset=['inn_key', 'agr_id_key'])
    if 'n_agr' in lk.columns:
        lk['_n_agr'] = lk['n_agr'].astype(str)
        agg = lk.groupby(['inn_key', 'agr_id_key'], as_index=False).agg(
            rows=('trx_cnt', 'size'),
            n_agr_nunique=('_n_agr', 'nunique'),
            trx_cnt_max=('trx_cnt', 'max'),
            trx_cnt_sum=('trx_cnt', 'sum'),
            trx_sum_max=('trx_sum', 'max'),
            trx_sum_sum=('trx_sum', 'sum'),
            ops_max=('commission_from_ops', 'max'),
            ops_sum=('commission_from_ops', 'sum'),
        )
    else:
        agg = lk.groupby(['inn_key', 'agr_id_key'], as_index=False).agg(
            rows=('trx_cnt', 'size'),
            trx_cnt_max=('trx_cnt', 'max'),
            trx_cnt_sum=('trx_cnt', 'sum'),
            trx_sum_max=('trx_sum', 'max'),
            trx_sum_sum=('trx_sum', 'sum'),
            ops_max=('commission_from_ops', 'max'),
            ops_sum=('commission_from_ops', 'sum'),
        )
        agg['n_agr_nunique'] = np.nan
    agg['cnt_max_ne_sum'] = agg['trx_cnt_max'].fillna(0).ne(agg['trx_cnt_sum'].fillna(0))
    return lk, agg


print('=== строки final_df на ключ inn+agr ===')
grain_rows = []
for month in ['2026-04', '2026-05', '2026-06']:
    _, g = _lake_raw_grain(month)
    vc = g['rows'].value_counts().sort_index()
    print(f'\n{month}: keys={len(g):,}  keys_rows>1={(g["rows"]>1).sum():,}  keys_max!=sum={int(g["cnt_max_ne_sum"].sum()):,}')
    print(vc.head(8).to_string())
    grain_rows.append({
        'report_month': month,
        'keys': len(g),
        'keys_multi_row': int((g['rows'] > 1).sum()),
        'keys_max_ne_sum': int(g['cnt_max_ne_sum'].sum()),
        'multi_row_share_pct': round(100.0 * (g['rows'] > 1).mean(), 2),
        'extra_cnt_if_sum_minus_max': float((g['trx_cnt_sum'] - g['trx_cnt_max']).fillna(0).sum()),
    })
display(pd.DataFrame(grain_rows))

# exact% если QC брать sum, не max — только май vs Excel
path, header = EXCEL_BY_MONTH[TARGET]
ex = _excel_full(path, header)
_, g_may = _lake_raw_grain(TARGET)
cmp_sum = g_may.merge(ex, on=['inn_key', 'agr_id_key'], how='inner')
n = len(cmp_sum)
exact_max = int(cmp_sum['trx_cnt_max'].fillna(0).eq(cmp_sum['trx_cnt_excel'].fillna(0)).sum())
exact_sum = int(cmp_sum['trx_cnt_sum'].fillna(0).eq(cmp_sum['trx_cnt_excel'].fillna(0)).sum())
print('\n=== май exact trx_cnt на ключах both ===')
print(f'max (как боевой QC): {100.0 * exact_max / n:.2f}%  ({exact_max:,}/{n:,})')
print(f'sum (гипотеза дробления): {100.0 * exact_sum / n:.2f}%  ({exact_sum:,}/{n:,})')

if 'may_both' in globals():
    j = may_both.merge(
        g_may[['inn_key', 'agr_id_key', 'rows', 'n_agr_nunique', 'trx_cnt_max', 'trx_cnt_sum', 'cnt_max_ne_sum']],
        on=['inn_key', 'agr_id_key'],
        how='left',
    )
    mis_j = j.loc[~j['cnt_exact']].copy()
    print('\n=== mismatch мая: сколько строк lake на ключе ===')
    print(mis_j['rows'].value_counts().sort_index().to_string())
    print('mismatch где max!=sum:', int(mis_j['cnt_max_ne_sum'].fillna(False).sum()))
    print('mismatch с n_agr>1:', int((mis_j['n_agr_nunique'] > 1).sum()) if mis_j['n_agr_nunique'].notna().any() else 'n_agr нет')

print('\n=== как читать ===')
print('Если sum поднимает exact к 95%+ — виноват QC max + дробление строк мая, не ODS.')
print('Если exact почти не сдвинулся и rows=1 на mismatch — в самой строке final_df число меньше Excel. Дальше фильтры section 05.')


## 7) Что Excel мая считает лишним относительно section 05

Блок 6 закрыл дробление: `max`/`sum` оба 55.62%, у 2 860 mismatch одна строка. В `final_df` число **меньше Excel на том же договоре**.

Озеро в апреле с тем же SQL даёт 99.5%. Значит либо Excel мая шире (реверсы, не-S01, не-RSHB, другая дата), либо в ODS мая на крупных договорах не хватает ~3%.

Один месяц, без `trx_int`. Считает по `n_agr` четыре варианта и сверяет с Excel. Апрель — контроль: вариант, который «чинит» май, не должен ломать апрель.


In [ ]:
def sql_trx_variants(ym):
    """По n_agr: strict (section 05) и три ослабления. Без trx_int."""
    ms, me = month_bounds(ym)
    return f"""
    with sa_agr as (
      select distinct cast(a.n_agr as string) as n_agr
      from ods_alpha.scd1_agreements a
      where upper(trim(cast(a.acq_class as string))) = 'SA'
        and cast(a.d_valid_from as date) <= cast('{me}' as date)
        and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{ms}' as date))
        and coalesce(a.ods_deleted_flg, '0') <> '1'
    ),
    fiid_rshb as (
      select distinct cast(fa.c_fiid as string) as c_fiid
      from ods_alpha.scd1_base24_fiids fa
      where coalesce(cast(fa.c_fiid_grp as string), 'UNKNOWN') = 'RSHB'
    ),
    trx as (
      select
        cast(t.n_trx as string) as n_trx,
        cast(t.c_trx_type as string) as c_trx_type,
        coalesce(cast(t.cf_trx_stat as string), '') as cf_trx_stat,
        cast(t.n_amt_src as double) as n_amt_src,
        case when fr.c_fiid is not null then 1 else 0 end as is_rshb
      from ods_alpha.scd1_trx t
      left join fiid_rshb fr on fr.c_fiid = cast(t.c_fiid_acq as string)
      where cast(t.d_trx_orig as timestamp) >= cast('{ms}' as timestamp)
        and cast(t.d_trx_orig as timestamp) < cast(date_add(cast('{me}' as date), 1) as timestamp)
        and t.c_nter is not null
        and coalesce(t.ods_deleted_flg, '0') <> '1'
        and t.c_trx_class = 'SA'
    ),
    ta as (
      select
        cast(a.n_trx as string) as n_trx,
        cast(a.n_agr as string) as n_agr,
        max(coalesce(cast(a.n_amt_tax as double), 0.0)) as n_amt_tax
      from ods_alpha.scd1_trx_acq a
      join sa_agr s on s.n_agr = cast(a.n_agr as string)
      where coalesce(a.ods_deleted_flg, '0') <> '1'
      group by cast(a.n_trx as string), cast(a.n_agr as string)
    )
    select
      ta.n_agr,
      count(distinct case
        when trx.is_rshb = 1 and trx.c_trx_type = 'S01' and trx.cf_trx_stat <> 'R'
        then trx.n_trx end) as cnt_strict,
      count(distinct case
        when trx.is_rshb = 1 and trx.c_trx_type = 'S01'
        then trx.n_trx end) as cnt_plus_rev,
      count(distinct case
        when trx.is_rshb = 1 and trx.cf_trx_stat <> 'R'
        then trx.n_trx end) as cnt_all_types,
      count(distinct case
        when trx.c_trx_type = 'S01' and trx.cf_trx_stat <> 'R'
        then trx.n_trx end) as cnt_all_fiid,
      sum(case
        when trx.is_rshb = 1 and trx.c_trx_type = 'S01' and trx.cf_trx_stat <> 'R'
        then trx.n_amt_src else 0 end) as sum_strict,
      sum(case
        when trx.is_rshb = 1 and trx.c_trx_type = 'S01'
        then trx.n_amt_src else 0 end) as sum_plus_rev,
      sum(case
        when trx.is_rshb = 1 and trx.c_trx_type = 'S01' and trx.cf_trx_stat <> 'R'
        then ta.n_amt_tax else 0 end) as ops_strict,
      sum(case
        when trx.is_rshb = 1 and trx.c_trx_type = 'S01'
        then ta.n_amt_tax else 0 end) as ops_plus_rev
    from ta
    join trx on trx.n_trx = ta.n_trx
    group by ta.n_agr
    """


def _score_variant(both, lake_col, excel_col='trx_cnt_excel'):
    lv = pd.to_numeric(both[lake_col], errors='coerce').fillna(0)
    ev = pd.to_numeric(both[excel_col], errors='coerce').fillna(0)
    exact = lv.eq(ev)
    n = len(both)
    return {
        'variant': lake_col,
        'exact_pct': round(100.0 * exact.mean(), 2) if n else None,
        'exact_n': int(exact.sum()),
        'keys': n,
        'lake_sum': float(lv.sum()),
        'excel_sum': float(ev.sum()),
        'ratio': float(lv.sum() / ev.sum()) if ev.sum() else None,
        'median_ratio': float((lv / ev.replace(0, np.nan)).median()),
    }


def _excel_vs_variants(ym, var_df):
    path, header = EXCEL_BY_MONTH[ym]
    ex = _excel_full(path, header)
    raw = load_lake_month(ym)
    if raw is None or raw.empty:
        raise RuntimeError(f'no final_df {ym}')
    lk = raw.copy()
    if 'agr_id' not in lk.columns and 'n_agr' in lk.columns:
        lk['agr_id'] = lk['n_agr']
    lk['inn_key'] = lk['inn'].map(normalize_inn_q1)
    lk['agr_id_key'] = lk['agr_id'].map(normalize_agr_q1)
    lk['n_agr_key'] = lk['n_agr'].map(normalize_agr_q1) if 'n_agr' in lk.columns else lk['agr_id_key']
    key_map = (
        lk.dropna(subset=['inn_key', 'agr_id_key', 'n_agr_key'])
          .groupby(['inn_key', 'agr_id_key'], as_index=False)
          .agg(n_agr_key=('n_agr_key', 'first'))
    )
    vv = var_df.copy()
    vv['n_agr_key'] = vv['n_agr'].map(normalize_agr_q1)
    both = (
        key_map.merge(ex, on=['inn_key', 'agr_id_key'], how='inner')
               .merge(vv, on='n_agr_key', how='left')
    )
    print(f'{ym}: keys both excel+final_df={len(both):,}  с вариантами ODS={(both["cnt_strict"].notna()).sum():,}')
    rows = [_score_variant(both, c) for c in ['cnt_strict', 'cnt_plus_rev', 'cnt_all_types', 'cnt_all_fiid']]
    # final_df as reference
    if 'trx_cnt' in lk.columns:
        fd = (
            lk.dropna(subset=['inn_key', 'agr_id_key'])
              .groupby(['inn_key', 'agr_id_key'], as_index=False)
              .agg(cnt_final_df=('trx_cnt', 'max'))
        )
        both2 = both.merge(fd, on=['inn_key', 'agr_id_key'], how='left')
        rows.append(_score_variant(both2, 'cnt_final_df'))
        both = both2
    stats = pd.DataFrame(rows)
    display(stats)
    return both, stats


print('Impala: май по n_agr, 4 варианта фильтров (один запрос)')
may_var = run_sql(sql_trx_variants('2026-05'), '=== 7. Май variants по n_agr ===')
may_both_var, may_var_stats = _excel_vs_variants('2026-05', may_var)

print('\nКонтроль апрель — тот же SQL')
apr_var = run_sql(sql_trx_variants('2026-04'), '=== 7. Апрель variants по n_agr ===')
apr_both_var, apr_var_stats = _excel_vs_variants('2026-04', apr_var)

print('\n=== какой вариант ближе к Excel ===')
side = may_var_stats[['variant', 'exact_pct', 'ratio']].merge(
    apr_var_stats[['variant', 'exact_pct', 'ratio']],
    on='variant', suffixes=('_may', '_apr'),
)
display(side)

print('\n=== как читать ===')
print('Ищем строку, где exact_pct_may >> 55% и exact_pct_apr остаётся ~99%.')
print('cnt_plus_rev чинит май, апрель падает → Excel мая начал включать реверсы.')
print('cnt_all_types чинит май → Excel мая взял не только S01.')
print('cnt_all_fiid чинит май → Excel без фильтра RSHB.')
print('Ни один не чинит, cnt_strict ≈ cnt_final_df < Excel → в ODS на договоре нет этих операций (другая дата или Excel из другого контура).')

best = side.sort_values('exact_pct_may', ascending=False).iloc[0]
print(f'\\nЛучший для мая: {best["variant"]}  may={best["exact_pct_may"]}%  apr={best["exact_pct_apr"]}%')


## 8) Дата проводки и сироты без `trx_acq`

Блок 7 закрыл фильтры 05. Апрель: `cnt_strict` = 99.53% (Excel = витрина). Май: ослабления только **хуже**. `cnt_strict` ≈ `final_df` — недобор уже в ODS на договоре, не в поздних шагах тетрадки.

Лишние ~174 тыс. в Excel мая — **не** реверсы и не другие типы на том же `n_agr` + `d_trx_orig`.

Осталось:

1. Excel режет месяц по другой дате (`trx_acq.d_trx_local` vs `d_trx_orig`).
2. Часть мая в `scd1_trx` есть, а `trx_acq` нет — витрина их не видит.

Снова май + апрель-контроль. Без `trx_int`.


In [ ]:
def sql_trx_date_variants(ym):
    ms, me = month_bounds(ym)
    return f"""
    with sa_agr as (
      select distinct cast(a.n_agr as string) as n_agr
      from ods_alpha.scd1_agreements a
      where upper(trim(cast(a.acq_class as string))) = 'SA'
        and cast(a.d_valid_from as date) <= cast('{me}' as date)
        and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{ms}' as date))
        and coalesce(a.ods_deleted_flg, '0') <> '1'
    ),
    fiid_rshb as (
      select distinct cast(fa.c_fiid as string) as c_fiid
      from ods_alpha.scd1_base24_fiids fa
      where coalesce(cast(fa.c_fiid_grp as string), 'UNKNOWN') = 'RSHB'
    ),
    trx_ok as (
      select cast(t.n_trx as string) as n_trx
      from ods_alpha.scd1_trx t
      join fiid_rshb fr on fr.c_fiid = cast(t.c_fiid_acq as string)
      where t.c_nter is not null
        and coalesce(t.ods_deleted_flg, '0') <> '1'
        and t.c_trx_class = 'SA'
        and t.c_trx_type = 'S01'
        and coalesce(t.cf_trx_stat, '') <> 'R'
    ),
    ta as (
      select
        cast(a.n_trx as string) as n_trx,
        cast(a.n_agr as string) as n_agr,
        to_date(cast(a.d_trx_local as timestamp)) as dt_local
      from ods_alpha.scd1_trx_acq a
      join sa_agr s on s.n_agr = cast(a.n_agr as string)
      join trx_ok t on t.n_trx = cast(a.n_trx as string)
      where coalesce(a.ods_deleted_flg, '0') <> '1'
    ),
    orig as (
      select cast(t.n_trx as string) as n_trx
      from ods_alpha.scd1_trx t
      join fiid_rshb fr on fr.c_fiid = cast(t.c_fiid_acq as string)
      where cast(t.d_trx_orig as timestamp) >= cast('{ms}' as timestamp)
        and cast(t.d_trx_orig as timestamp) < cast(date_add(cast('{me}' as date), 1) as timestamp)
        and t.c_nter is not null
        and coalesce(t.ods_deleted_flg, '0') <> '1'
        and t.c_trx_class = 'SA'
        and t.c_trx_type = 'S01'
        and coalesce(t.cf_trx_stat, '') <> 'R'
    )
    select
      ta.n_agr,
      count(distinct case when o.n_trx is not null then ta.n_trx end) as cnt_orig,
      count(distinct case
        when ta.dt_local >= cast('{ms}' as date)
         and ta.dt_local <  date_add(cast('{me}' as date), 1)
        then ta.n_trx end) as cnt_local,
      count(distinct case
        when o.n_trx is not null
          or (
            ta.dt_local >= cast('{ms}' as date)
            and ta.dt_local < date_add(cast('{me}' as date), 1)
          )
        then ta.n_trx end) as cnt_orig_or_local
    from ta
    left join orig o on o.n_trx = ta.n_trx
    group by ta.n_agr
    """


def sql_orphan_acq(ym):
    ms, me = month_bounds(ym)
    return f"""
    with fiid_rshb as (
      select distinct cast(fa.c_fiid as string) as c_fiid
      from ods_alpha.scd1_base24_fiids fa
      where coalesce(cast(fa.c_fiid_grp as string), 'UNKNOWN') = 'RSHB'
    ),
    trx as (
      select cast(t.n_trx as string) as n_trx
      from ods_alpha.scd1_trx t
      join fiid_rshb fr on fr.c_fiid = cast(t.c_fiid_acq as string)
      where cast(t.d_trx_orig as timestamp) >= cast('{ms}' as timestamp)
        and cast(t.d_trx_orig as timestamp) < cast(date_add(cast('{me}' as date), 1) as timestamp)
        and t.c_nter is not null
        and coalesce(t.ods_deleted_flg, '0') <> '1'
        and t.c_trx_class = 'SA'
        and t.c_trx_type = 'S01'
        and coalesce(t.cf_trx_stat, '') <> 'R'
    )
    select
      '{ym}' as report_month,
      count(*) as trx_rshb_s01,
      count(a.n_trx) as with_trx_acq,
      count(*) - count(a.n_trx) as orphan_no_acq,
      round(100.0 * (count(*) - count(a.n_trx)) / count(*), 3) as orphan_pct
    from trx t
    left join (
      select cast(n_trx as string) as n_trx
      from ods_alpha.scd1_trx_acq
      where coalesce(ods_deleted_flg, '0') <> '1'
      group by 1
    ) a on a.n_trx = t.n_trx
    """


def _merge_excel_ods(ym, var_df):
    path, header = EXCEL_BY_MONTH[ym]
    ex = _excel_full(path, header)
    raw = load_lake_month(ym)
    lk = raw.copy()
    if 'agr_id' not in lk.columns and 'n_agr' in lk.columns:
        lk['agr_id'] = lk['n_agr']
    lk['inn_key'] = lk['inn'].map(normalize_inn_q1)
    lk['agr_id_key'] = lk['agr_id'].map(normalize_agr_q1)
    lk['n_agr_key'] = lk['n_agr'].map(normalize_agr_q1) if 'n_agr' in lk.columns else lk['agr_id_key']
    key_map = (
        lk.dropna(subset=['inn_key', 'agr_id_key', 'n_agr_key'])
          .groupby(['inn_key', 'agr_id_key'], as_index=False)
          .agg(n_agr_key=('n_agr_key', 'first'))
    )
    vv = var_df.copy()
    vv['n_agr_key'] = vv['n_agr'].map(normalize_agr_q1)
    both = key_map.merge(ex, on=['inn_key', 'agr_id_key'], how='inner').merge(vv, on='n_agr_key', how='left')
    print(f'{ym}: both={len(both):,}  ods_hit={both["n_agr"].notna().sum():,}')
    return both


print('Сироты trx без trx_acq (апрель vs май) — лёгкий запрос')
orph = pd.concat([
    run_sql(sql_orphan_acq('2026-04'), '=== 8. orphans 2026-04 ==='),
    run_sql(sql_orphan_acq('2026-05'), '=== 8. orphans 2026-05 ==='),
    run_sql(sql_orphan_acq('2026-06'), '=== 8. orphans 2026-06 ==='),
], ignore_index=True)
display(orph)

print('\nImpala: май по дате orig vs local')
may_date = run_sql(sql_trx_date_variants('2026-05'), '=== 8. Май orig/local по n_agr ===')
may_d_both = _merge_excel_ods('2026-05', may_date)
print('май exact по дате:')
display(pd.DataFrame([
    _score_variant(may_d_both, c)
    for c in ['cnt_orig', 'cnt_local', 'cnt_orig_or_local']
    if c in may_d_both.columns
]))

print('\nКонтроль апрель')
apr_date = run_sql(sql_trx_date_variants('2026-04'), '=== 8. Апрель orig/local по n_agr ===')
apr_d_both = _merge_excel_ods('2026-04', apr_date)
print('апрель exact по дате:')
display(pd.DataFrame([
    _score_variant(apr_d_both, c)
    for c in ['cnt_orig', 'cnt_local', 'cnt_orig_or_local']
    if c in apr_d_both.columns
]))

print('\n=== как читать ===')
print('orphan_pct май заметно выше апреля → дыра trx_acq в мае (озеро).')
print('cnt_local чинит май, апрель остаётся ~99% → Excel режет месяц по d_trx_local.')
print('cnt_orig_or_local чинит май, апрель падает → Excel шире по границе месяца.')
print('Оба ~55% → Excel мая не из scd1_trx+trx_acq (другой отчёт / контур).')


## 9) Один проблемный ИНН: все операции и сложение до Excel

Берём самый большой разрыв (блок 5):

- ИНН `2465008567`, договор `459854977840`
- озеро `1 230 107`, Excel `1 262 560`, не хватает **32 453** (~2.6%)

Сначала без Impala: не сидит ли этот хвост на **других договорах того же ИНН** в `final_df`.  
Потом ODS: все trx этого ИНН (широкое окно дат, без фильтра типа/статуса) и сложение срезов, пока не попадём в 1 262 560.


In [ ]:
TARGET_INN = '2465008567'
TARGET_AGR = '459854977840'
YM = '2026-05'
EXCEL_TARGET_CNT = 1_262_560  # подставится из файла, это ориентир

inn_k = normalize_inn_q1(TARGET_INN)
agr_k = normalize_agr_q1(TARGET_AGR)
print('target inn', inn_k, 'agr', agr_k)

# --- A) Excel + final_df: этот ключ и все договоры ИНН ---
path, header = EXCEL_BY_MONTH[YM]
ex_all = _excel_full(path, header)
ex_inn = ex_all.loc[ex_all['inn_key'] == inn_k].copy()
print('\n=== Excel мая по этому ИНН ===')
display(ex_inn)
print('Excel сумма trx_cnt по всем договорам ИНН:', float(ex_inn['trx_cnt_excel'].fillna(0).sum()))

raw = load_lake_month(YM)
lk = raw.copy()
if 'agr_id' not in lk.columns and 'n_agr' in lk.columns:
    lk['agr_id'] = lk['n_agr']
lk['inn_key'] = lk['inn'].map(normalize_inn_q1)
lk['agr_id_key'] = lk['agr_id'].map(normalize_agr_q1)
lk['n_agr_key'] = lk['n_agr'].map(normalize_agr_q1) if 'n_agr' in lk.columns else lk['agr_id_key']
for c in ['trx_cnt', 'trx_sum', 'commission_from_ops']:
    if c in lk.columns:
        lk[c] = pd.to_numeric(lk[c], errors='coerce')

lk_inn = lk.loc[lk['inn_key'] == inn_k, [
    c for c in ['inn_key', 'agr_id_key', 'n_agr', 'n_agr_key', 'trx_cnt', 'trx_sum', 'commission_from_ops']
    if c in lk.columns
]].copy()
print('\n=== final_df мая по этому ИНН ===')
display(lk_inn)
print('lake сумма trx_cnt по всем договорам ИНН:', float(lk_inn['trx_cnt'].fillna(0).sum()) if 'trx_cnt' in lk_inn.columns else None)

ex_row = ex_inn.loc[ex_inn['agr_id_key'] == agr_k]
lk_row = lk_inn.loc[lk_inn['agr_id_key'] == agr_k]
excel_cnt = float(ex_row['trx_cnt_excel'].sum()) if len(ex_row) else np.nan
lake_cnt = float(lk_row['trx_cnt'].sum()) if len(lk_row) and 'trx_cnt' in lk_row.columns else np.nan
gap = excel_cnt - lake_cnt
print(f'\nключ: excel={excel_cnt:,.0f}  lake={lake_cnt:,.0f}  gap={gap:,.0f}')
other_agr_cnt = float(lk_inn.loc[lk_inn['agr_id_key'] != agr_k, 'trx_cnt'].fillna(0).sum()) if 'trx_cnt' in lk_inn.columns else 0
print(f'другие договоры этого ИНН в lake: {other_agr_cnt:,.0f}')
if other_agr_cnt and abs(other_agr_cnt - gap) < 50:
    print('→ Excel, похоже, сложил ВСЕ договоры ИНН в одну строку.')
elif other_agr_cnt:
    print('→ На других договорах ИНН есть объём, но он не равен gap. Смотрим ODS.')
else:
    print('→ Других договоров ИНН в final_df нет (или нули). Хвост не из соседнего agr витрины.')

# --- B) ODS: все trx ИНН, широкое окно, без фильтра типа/статуса ---
ms, me = month_bounds(YM)
sql_one_inn = f"""
with inn_cmp as (
  select distinct cast(c.n_cmp as string) as n_cmp
  from ods_alpha.scd1_companies c
  where regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') = '{inn_k}'
    and coalesce(c.ods_deleted_flg, '0') <> '1'
),
agr as (
  select distinct
    cast(a.n_agr as string) as n_agr,
    cast(a.n_cmp_client as string) as n_cmp,
    cast(a.acq_class as string) as acq_class
  from ods_alpha.scd1_agreements a
  join inn_cmp c on c.n_cmp = cast(a.n_cmp_client as string)
  where coalesce(a.ods_deleted_flg, '0') <> '1'
),
fiid as (
  select
    cast(fa.c_fiid as string) as c_fiid,
    cast(fa.c_fiid_grp as string) as c_fiid_grp
  from ods_alpha.scd1_base24_fiids fa
),
ta as (
  select
    cast(x.n_trx as string) as n_trx,
    cast(x.n_agr as string) as n_agr,
    max(coalesce(cast(x.n_amt_tax as double), 0.0)) as n_amt_tax,
    max(to_date(cast(x.d_trx_local as timestamp))) as dt_local
  from ods_alpha.scd1_trx_acq x
  join agr g on g.n_agr = cast(x.n_agr as string)
  where coalesce(x.ods_deleted_flg, '0') <> '1'
  group by 1, 2
)
select
  ta.n_agr,
  g.acq_class,
  cast(t.c_trx_class as string) as c_trx_class,
  cast(t.c_trx_type as string) as c_trx_type,
  coalesce(cast(t.cf_trx_stat as string), '') as cf_trx_stat,
  coalesce(f.c_fiid_grp, 'UNKNOWN') as fiid_grp,
  to_date(cast(t.d_trx_orig as timestamp)) as dt_orig,
  ta.dt_local,
  count(*) as rows_cnt,
  count(distinct t.n_trx) as trx_cnt,
  sum(cast(t.n_amt_src as double)) as trx_sum,
  sum(ta.n_amt_tax) as ops_sum
from ta
join ods_alpha.scd1_trx t on cast(t.n_trx as string) = ta.n_trx
left join agr g on g.n_agr = ta.n_agr
left join fiid f on f.c_fiid = cast(t.c_fiid_acq as string)
where coalesce(t.ods_deleted_flg, '0') <> '1'
  and t.c_nter is not null
  and (
        (cast(t.d_trx_orig as timestamp) >= cast('2026-04-01' as timestamp)
         and cast(t.d_trx_orig as timestamp) <  cast('2026-07-01' as timestamp))
     or (ta.dt_local >= cast('2026-04-01' as date)
         and ta.dt_local <  cast('2026-07-01' as date))
  )
group by 1, 2, 3, 4, 5, 6, 7, 8
"""

print('\nImpala: все срезы trx этого ИНН, апр–июн (orig или local)')
inn_trx = run_sql(sql_one_inn, f'=== 9. INN {inn_k} trx slices ===')
inn_trx['dt_orig'] = pd.to_datetime(inn_trx['dt_orig'], errors='coerce')
inn_trx['dt_local'] = pd.to_datetime(inn_trx['dt_local'], errors='coerce')
inn_trx['orig_ym'] = inn_trx['dt_orig'].dt.strftime('%Y-%m')
inn_trx['local_ym'] = inn_trx['dt_local'].dt.strftime('%Y-%m')
inn_trx['is_target_agr'] = inn_trx['n_agr'].map(normalize_agr_q1).eq(agr_k)
inn_trx['is_s01'] = inn_trx['c_trx_type'].astype(str).eq('S01')
inn_trx['is_sa'] = inn_trx['c_trx_class'].astype(str).eq('SA')
inn_trx['is_rev'] = inn_trx['cf_trx_stat'].astype(str).eq('R')
inn_trx['is_rshb'] = inn_trx['fiid_grp'].astype(str).eq('RSHB')
inn_trx['orig_may'] = inn_trx['orig_ym'].eq(YM)
inn_trx['local_may'] = inn_trx['local_ym'].eq(YM)

print('\n=== свод по договору (target agr) ===')
tgt = inn_trx.loc[inn_trx['is_target_agr']].copy()
display(
    tgt.groupby(['orig_ym', 'c_trx_type', 'cf_trx_stat', 'fiid_grp'], as_index=False)
       .agg(trx_cnt=('trx_cnt', 'sum'), trx_sum=('trx_sum', 'sum'))
       .sort_values('trx_cnt', ascending=False)
       .head(30)
)


def _sum(mask):
    return float(inn_trx.loc[mask, 'trx_cnt'].fillna(0).sum())

strict = (
    inn_trx['is_target_agr'] & inn_trx['orig_may'] & inn_trx['is_sa']
    & inn_trx['is_s01'] & ~inn_trx['is_rev'] & inn_trx['is_rshb']
)
add_rows = [
    ('strict 05 (этот agr, orig май, SA/S01, не R, RSHB)', strict),
    ('+ реверсы', inn_trx['is_target_agr'] & inn_trx['orig_may'] & inn_trx['is_sa'] & inn_trx['is_s01'] & inn_trx['is_rshb']),
    ('+ все типы SA, не R, RSHB, orig май', inn_trx['is_target_agr'] & inn_trx['orig_may'] & inn_trx['is_sa'] & ~inn_trx['is_rev'] & inn_trx['is_rshb']),
    ('local май, SA/S01, не R, RSHB, этот agr', inn_trx['is_target_agr'] & inn_trx['local_may'] & inn_trx['is_sa'] & inn_trx['is_s01'] & ~inn_trx['is_rev'] & inn_trx['is_rshb']),
    ('orig ИЛИ local май, SA/S01, не R, RSHB, этот agr', inn_trx['is_target_agr'] & (inn_trx['orig_may'] | inn_trx['local_may']) & inn_trx['is_sa'] & inn_trx['is_s01'] & ~inn_trx['is_rev'] & inn_trx['is_rshb']),
    ('все agr ИНН, strict май', (~inn_trx['is_rev']) & inn_trx['orig_may'] & inn_trx['is_sa'] & inn_trx['is_s01'] & inn_trx['is_rshb']),
    ('все agr ИНН + любые тип/статус/fiid, orig май', inn_trx['orig_may']),
    ('все agr ИНН + любые, orig или local май', inn_trx['orig_may'] | inn_trx['local_may']),
]
recon = pd.DataFrame([
    {
        'slice': name,
        'trx_cnt': _sum(m),
        'vs_excel': _sum(m) - excel_cnt,
        'vs_lake': _sum(m) - lake_cnt,
        'hit_excel': abs(_sum(m) - excel_cnt) < 2,
    }
    for name, m in add_rows
])
print('\n=== сложение до Excel ===')
print(f'цель Excel = {excel_cnt:,.0f}   lake final_df = {lake_cnt:,.0f}   gap = {gap:,.0f}')
display(recon)

hits = recon.loc[recon['hit_excel']]
if len(hits):
    print('Попали в Excel срезом:')
    display(hits)
else:
    nearest = recon.assign(abs_d=recon['vs_excel'].abs()).sort_values('abs_d').head(3)
    print('Ни один срез не равен Excel. Ближе всего:')
    display(nearest)
    print('Если даже «все agr + любые тип/статус/дата май» < Excel — этих операций нет в scd1_trx/trx_acq. Excel из другого контура.')
    print('Если какой-то срез > Excel — Excel уже, чем полный ODS; ищем комбинацию флагов в своде выше.')
